# <center> Cell villages of adipogenic differentiation <center>
    
Here we will analyse the data produced in the cell village study

## Checking eQTLs

In [ ]:
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import pandas as pd
import matplotlib as mpl

mpl.rcParams['font.family'] = 'Latin Modern Roman' #Font for the plots


In [ ]:
BILLING_PROJECT_ID = os.environ['WORKSPACE_NAMESPACE']
WORKSPACE = os.environ['WORKSPACE_NAME']
bucket = os.environ['WORKSPACE_BUCKET']

#EXTERNAL_BUCKET = "gs://path_to_amsc_dataset_bucket/"
EXTERNAL_BUCKET = "gs://path_to_shared_bucket/shared_2023/"
NEW_EXTERNAL_BUCKET =  "gs://path_to_shared_bucket/shared_2024/"
CONTENT = !gsutil ls {EXTERNAL_BUCKET}
    
print("Billing project: " + BILLING_PROJECT_ID)
print("Workspace: " + WORKSPACE)
print("Bucket: " + bucket)
print("Files in the external bucket:", *CONTENT, sep='\n')


#! gsutil cp {bucket}/eQTL_villages/*  ./eQTL_villages

In [ ]:
def in_position_range(variant_id, start_pos, end_pos):
    """Check if variant position is within specified range"""
    match = re.match(r'^5:(\d+):[ATCG]+:[ATCG]+$', variant_id)
    if match:
        pos = int(match.group(1))
        return start_pos <= pos <= end_pos
    return False

def load_data(start_pos=56500000, end_pos=56530000):
    """Load and process all data files"""
    all_data = []
    gene_patterns = ['MAP3K1', 'SETD9', 'ANKRD55', 'MIER3']
    
    tissues = {
        'Adipogenic FFA': '2025_villages_eQTL_cis_nominal_cis_nominal_Adipogenic_FFA_Adipogenic_FFA.cis_qtl_pairs.chr5.parquet',
        'Adipogenic basal': '2025_villages_eQTL_v2_1_cis_nominal_Adipogenic_basal_Adipogenic_basal.cis_qtl_pairs.chr5.parquet',
        'SWAT FFA': '2025_villages_eQTL_v2_1_cis_nominal_SWAT_cells_FFA_SWAT_cells_FFA.cis_qtl_pairs.chr5.parquet',
        'Adipogenic Low Glucose': '2025_villages_eQTL_v2_1_cis_nominal_Adipogenic_cells_LowGlucose_Adipogenic_cells_LowGlucose.cis_qtl_pairs.chr5.parquet',
        'SWAT basal': '2025_villages_eQTL_v2_1_cis_nominal_SWAT_basal_SWAT_basal.cis_qtl_pairs.chr5.parquet'
    }
    
    for tissue_name, file in tissues.items():
        df = pd.read_parquet(f"./eQTL_villages/{file}")
        
        # Extract tissue type and stimulation
        tissue_type = 'Adipogenic' if 'Adipogenic' in tissue_name else 'SWAT'
        
        if 'FFA' in tissue_name:
            stimulation = 'FFA'
        elif 'basal' in tissue_name:
            stimulation = 'basal'
        elif 'Low Glucose' in tissue_name:
            stimulation = 'Low Glucose'
        else:
            stimulation = 'Unknown'
        
        # Filter for gene patterns and position range
        for pattern in gene_patterns:
            filtered_df = df[df['phenotype_id'].str.contains(pattern, case=False, regex=True)]
            var_filtered_df = filtered_df[filtered_df['variant_id'].apply(
                lambda x: in_position_range(x, start_pos, end_pos))]
            
            # Add data
            for _, row in var_filtered_df.iterrows():
                pos_match = re.match(r'^5:(\d+)', row['variant_id'])
                position = int(pos_match.group(1)) if pos_match else 0
                # Extract variant parts
                variant_parts = row['variant_id'].split(':')
                if len(variant_parts) >= 4:
                    variant_short = f"{variant_parts[1]}:{variant_parts[2]}:{variant_parts[3]}"
                else:
                    variant_short = variant_parts[1] if len(variant_parts) > 1 else "unknown"
                
                all_data.append({
                    'Gene': pattern,
                    'FullTissue': tissue_name,
                    'TissueType': tissue_type,
                    'Stimulation': stimulation,
                    'Variant': row['variant_id'],
                    'VariantShort': variant_short,
                    'Position': position,
                    '-log10(p-value)': -np.log10(row['pval_nominal']),
                    'Slope': row['slope'],
                    'GeneVariant': f"{pattern}-{variant_short}"  # Combined key
                })
    
    return pd.DataFrame(all_data)

def create_plot(df, hierarchy, title, filename):
    """Create plot with simplified approach and better labeling"""
    if df.empty:
        print(f"No data to plot for {filename}")
        return False
    
    # Sort dataframe according to hierarchy
    sorted_df = df.sort_values(by=hierarchy)
    
    # Create figure
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(35, 15), sharex=True)
    
    # Create color maps
    gene_colors = {gene: plt.cm.tab10(i) for i, gene in 
                  enumerate(sorted_df['Gene'].unique())}
    
    # Plot bars
    positions = range(len(sorted_df))
    
    # Plot p-values
    ax1.bar(positions, sorted_df['-log10(p-value)'], color=[gene_colors[g] for g in sorted_df['Gene']])
    ax1.set_ylabel('-log10(p-value)', fontsize=16)
    ax1.set_ylim(0,4.0)
    ax1.set_title(f'{title} - eQTL p-values', fontsize=16)
    ax1.axhline(-np.log10(0.05), color='red', linestyle='--', label='p=0.05')
    ax1.legend()
    
    # Plot slopes
    ax2.bar(positions, sorted_df['Slope'], color=[gene_colors[g] for g in sorted_df['Gene']])
    ax2.set_ylabel('Slope (Effect Size)', fontsize=16)
    ax2.set_title(f'{title} - eQTL Effect Sizes', fontsize=16)
    ax2.axhline(0, color='black', linestyle='-', alpha=0.3)
    
    # Add group labels for the first hierarchy level
    current_group = None
    group_start = 0
    
    for i, group in enumerate(sorted_df[hierarchy[0]]):
        if group != current_group:
            # Add separator line for previous group
            if current_group is not None:
                ax1.axvline(i - 0.5, color='black', linestyle='-', alpha=0.5)
                ax2.axvline(i - 0.5, color='black', linestyle='-', alpha=0.5)
                
                # Add bold label for previous group
                mid_point = (group_start + i - 1) / 2
                ax1.text(mid_point, ax1.get_ylim()[1] * 0.9, str(current_group),
                        horizontalalignment='center', fontsize=12, fontweight='bold')
            
            # Update tracking variables
            current_group = group
            group_start = i
    
    # Add label for the last group
    if current_group is not None:
        mid_point = (group_start + len(sorted_df) - 1) / 2
        ax1.text(mid_point, ax1.get_ylim()[1] * 0.9, str(current_group),
                horizontalalignment='center', fontsize=12, fontweight='bold')
    
    # Create labels for bars (excluding the first hierarchy level)
    labels = []
    for _, row in sorted_df.iterrows():
        # Create label parts based on hierarchy (skip first level)
        label_parts = []
        for col in hierarchy[1:]:
            if col == 'Variant':
                label_parts.append(row['VariantShort'])
            elif col == 'VariantShort':
                label_parts.append(row['VariantShort'])
            else:
                label_parts.append(str(row[col]))
        
        # Join with dashes
        labels.append('-'.join(label_parts))
    
    # Set labels
    ax2.set_xticks(positions)
    ax2.set_xticklabels(labels, rotation=90)
    
    # Add gene legend
    legend_handles = [plt.Rectangle((0,0), 1, 1, color=gene_colors[gene]) 
                     for gene in gene_colors]
    fig.legend(legend_handles, gene_colors.keys(), title='Genes', 
              loc='upper center', bbox_to_anchor=(0.5, 0.98), ncol=len(gene_colors))
    
    plt.tight_layout()
    plt.subplots_adjust(top=0.9, bottom=0.2)  # More space for labels
    plt.savefig(f'{filename}.png', dpi=200)
    plt.show()
    
    print(f"Created {filename}.png")
    return True

def create_adipogenic_variant_gene_plot(df, filename="eQTL_adipogenic_genevar"):
    """Create plot showing all adipogenic results grouped by gene-variant pairs"""
    # Filter for adipogenic cells only
    adipo_df = df[df['TissueType'] == 'Adipogenic'].copy()
    
    if adipo_df.empty:
        print("No Adipogenic data found")
        return False
    
    # Sort by gene-variant pairs and stimulation
    sorted_df = adipo_df.sort_values(['GeneVariant', 'Stimulation'])
    
    # Create figure
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(25, 10), sharex=True)
    
    # Create color map for stimulation conditions
    stim_colors = {
        'FFA': 'green',
        'basal': 'purple',
        'Low Glucose': 'orange'
    }
    
    # Plot bars
    positions = range(len(sorted_df))
    
    # Plot p-values
    ax1.bar(positions, sorted_df['-log10(p-value)'], 
            color=[stim_colors.get(s, 'gray') for s in sorted_df['Stimulation']])
    ax1.set_ylabel('-log10(p-value)')
    ax1.set_title(f'Adipogenic Cells: eQTL p-values by Gene-Variant Pairs')
    ax1.axhline(-np.log10(0.05), color='red', linestyle='--', label='p=0.05')
    
    # Plot slopes
    ax2.bar(positions, sorted_df['Slope'], 
            color=[stim_colors.get(s, 'gray') for s in sorted_df['Stimulation']])
    ax2.set_ylabel('Slope (Effect Size)')
    ax2.set_title(f'Adipogenic Cells: eQTL Effect Sizes by Gene-Variant Pairs')
    ax2.axhline(0, color='black', linestyle='-', alpha=0.3)
    
    # Add group labels for gene-variant pairs
    current_group = None
    group_start = 0
    
    for i, genevar in enumerate(sorted_df['GeneVariant']):
        if genevar != current_group:
            # Add separator line for previous group
            if current_group is not None:
                ax1.axvline(i - 0.5, color='black', linestyle='-', alpha=0.5)
                ax2.axvline(i - 0.5, color='black', linestyle='-', alpha=0.5)
                
                # Add bold label for previous group
                mid_point = (group_start + i - 1) / 2
                ax1.text(mid_point, ax1.get_ylim()[1] * 0.9, str(current_group),
                        horizontalalignment='center', fontsize=12, fontweight='bold')
            
            # Update tracking variables
            current_group = genevar
            group_start = i
    
    # Add label for the last group
    if current_group is not None:
        mid_point = (group_start + len(sorted_df) - 1) / 2
        ax1.text(mid_point, ax1.get_ylim()[1] * 0.9, str(current_group),
                horizontalalignment='center', fontsize=12, fontweight='bold')
    
    # Create labels for stimulation conditions
    labels = [row['Stimulation'] for _, row in sorted_df.iterrows()]
    
    # Set labels
    ax2.set_xticks(positions)
    ax2.set_xticklabels(labels, rotation=90)
    
    # Add stimulation legend
    legend_handles = [plt.Rectangle((0,0), 1, 1, color=color) 
                     for stim, color in stim_colors.items()]
    fig.legend(legend_handles, stim_colors.keys(), title='Stimulation Conditions', 
              loc='upper center', bbox_to_anchor=(0.5, 0.98), ncol=len(stim_colors))
    
    plt.tight_layout()
    plt.subplots_adjust(top=0.9, bottom=0.2)
    plt.savefig(f'{filename}.png', dpi=200)
    plt.show()
    
    print(f"Created {filename}.png")
    return True

def create_summary_stats_plot(df, filename_prefix="eQTL_summary"):
    """Create plot showing mean and std across conditions, ordered by variant position"""
    # Calculate stats for Adipogenic cells across stimulations
    adipo_df = df[df['TissueType'] == 'Adipogenic'].copy()
    
    if adipo_df.empty:
        print("No Adipogenic data found")
        return False
    
    # Calculate mean and std by gene-variant pairs
    stats_df = adipo_df.groupby(['GeneVariant', 'Position']).agg({
        '-log10(p-value)': ['mean', 'std'],
        'Slope': ['mean', 'std']
    }).reset_index()
    
    # Flatten column names
    stats_df.columns = [f"{col[0]}_{col[1]}" if col[1] else col[0] for col in stats_df.columns]
    
    # Extract gene for coloring
    stats_df['Gene'] = stats_df['GeneVariant'].apply(lambda x: x.split('-')[0])
    
    # Sort by position instead of gene
    stats_df = stats_df.sort_values('Position')
    
    # Create figure
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(20, 15), sharex=True)
    
    # Create color map for genes
    gene_patterns = df['Gene'].unique()
    gene_colors = {gene: plt.cm.tab10(i) for i, gene in enumerate(gene_patterns)}
    
    # Plot positions
    positions = range(len(stats_df))
    
    # Plot mean p-values with error bars
    ax1.bar(positions, stats_df['-log10(p-value)_mean'], 
            yerr=stats_df['-log10(p-value)_std'],
            color=[gene_colors[g] for g in stats_df['Gene']])
    ax1.set_ylabel('Mean -log10(p-value) ± std')
    ax1.set_title('Adipogenic Cells: Mean eQTL p-values Across Stimulations (Ordered by Position)')
    ax1.axhline(-np.log10(0.05), color='red', linestyle='--', label='p=0.05')
    
    # Plot mean slopes with error bars
    ax2.bar(positions, stats_df['Slope_mean'], 
            yerr=stats_df['Slope_std'],
            color=[gene_colors[g] for g in stats_df['Gene']])
    ax2.set_ylabel('Mean Slope (Effect Size) ± std')
    ax2.set_title('Adipogenic Cells: Mean eQTL Effect Sizes Across Stimulations (Ordered by Position)')
    ax2.axhline(0, color='black', linestyle='-', alpha=0.3)
    
    # Add position information to labels
    labels = [f"{gv} ({pos})" for gv, pos in zip(stats_df['GeneVariant'], stats_df['Position'])]
    
    # Set x-labels to gene-variant identifiers with positions
    ax2.set_xticks(positions)
    ax2.set_xticklabels(labels, rotation=90)
    
    # Add gene legend
    legend_handles = [plt.Rectangle((0,0), 1, 1, color=gene_colors[gene]) 
                     for gene in gene_colors]
    fig.legend(legend_handles, gene_colors.keys(), title='Genes', 
              loc='upper center', bbox_to_anchor=(0.5, 0.98), ncol=len(gene_colors))
    
    plt.tight_layout()
    plt.subplots_adjust(top=0.9, bottom=0.2)
    plt.savefig(f'{filename_prefix}_mean_std_by_position.png', dpi=150)
    plt.show()
    
    print(f"Created {filename_prefix}_mean_std_by_position.png")
    return True

def main():
    # Load data
    plot_df = load_data()
    
    # Define hierarchies to plot
    hierarchies = [
        (['TissueType', 'Stimulation', 'Gene', 'VariantShort'], 
         'Tissue > Stimulation > Gene > Variant', 
         'eQTL_tissue_stim_gene_var'),
        
        (['Gene', 'TissueType', 'Stimulation', 'VariantShort'], 
         'Gene > Tissue > Stimulation > Variant', 
         'eQTL_gene_tissue_stim_var'),
         
        (['VariantShort', 'Gene', 'TissueType', 'Stimulation'], 
         'Variant > Gene > Tissue > Stimulation', 
         'eQTL_var_gene_tissue_stim'),
        
        (['GeneVariant', 'TissueType', 'Stimulation'], 
         'Gene-Variant Pairs > Tissue > Stimulation', 
         'eQTL_genevar_tissue_stim')
    ]
    
    # Create plots for each hierarchy
    for columns, title, filename in hierarchies:
        create_plot(plot_df, columns, title, filename)
    
    # Create adipogenic-specific plots
    create_adipogenic_variant_gene_plot(plot_df)
    create_summary_stats_plot(plot_df)
    
    print("All visualizations completed!")

if __name__ == "__main__":
    main()

## Checking TF motifs enriched in ATAC peaks for FFA positive, adipogenic cells

In [ ]:
AMSC_village_bucket = "gs://path_to_village_bucket/2025_villages/atac_motif_enrichments/"
csv_files = ! gsutil ls {AMSC_village_bucket}

for file in csv_files:
    if "csv" in file:
        print(file)
        df = pd.read_csv(file).sort_values("p-value", ascending=True)
        print(df.head(30))

**Trans-QTLs:**
    
The trans-QTL analysis is completely underpowered and was not included.

In [ ]:
import pandas as pd

files = ['gs://path_to_village_bucket/2025_villages/eQTL/v2_1/trans/Adipogenic_basal.trans_qtl_pairs.parquet',
         'gs://path_to_village_bucket/2025_villages/eQTL/v2_1/trans/Adipogenic_cells_LowGlucose.trans_qtl_pairs.parquet',
         'gs://path_to_village_bucket/2025_villages/eQTL/v2_1/trans/Adipogenic_FFA.trans_qtl_pairs.parquet']

for file in files:

    df = pd.read_parquet(file)

    chrom, start, end = ('5', 56500000, 56530000)

    # Split the variant_id column into its components
    df[['chr', 'pos', 'ref', 'alt']] = df['variant_id'].str.split(':', expand=True)

    # Convert the 'pos' column to integer for numerical comparison
    df['pos'] = pd.to_numeric(df['pos'])

    # Filter the DataFrame based on chromosome and position
    filtered_df = df[(df['chr'] == chrom) & (df['pos'] >= start) & (df['pos'] <= end)].copy()

    # Drop the temporary split columns
    filtered_df = filtered_df.drop(columns=['chr', 'pos', 'ref', 'alt'])

    print(file,'\n', sorted(filtered_df['phenotype_id'].unique()))